In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import cv2 
import os 
import torch
import matplotlib.pyplot as plt


if "PYOPENGL_PLATFORM" not in os.environ:
    os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np

import pyrender

from typing import Optional, List

# !mkdir -p data/cow_mesh
# !wget -P data/cow_mesh https://dl.fbaipublicfiles.com/pytorch3d/data/cow_mesh/cow.obj
# !wget -P data/cow_mesh https://dl.fbaipublicfiles.com/pytorch3d/data/cow_mesh/cow.mtl
# !wget -P data/cow_mesh https://dl.fbaipublicfiles.com/pytorch3d/data/cow_mesh/cow_texture.png


import trimesh



In [ ]:
def create_raymond_lights() -> List[pyrender.Node]:
    """
    Return raymond light nodes for the scene.
    """
    thetas = np.pi * np.array([1.0 / 6.0, 1.0 / 6.0, 1.0 / 6.0])
    phis = np.pi * np.array([0.0, 2.0 / 3.0, 4.0 / 3.0])

    nodes = []

    for phi, theta in zip(phis, thetas):
        xp = np.sin(theta) * np.cos(phi)
        yp = np.sin(theta) * np.sin(phi)
        zp = np.cos(theta)

        z = np.array([xp, yp, zp])
        z = z / np.linalg.norm(z)
        x = np.array([-z[1], z[0], 0.0])
        if np.linalg.norm(x) == 0:
            x = np.array([1.0, 0.0, 0.0])
        x = x / np.linalg.norm(x)
        y = np.cross(z, x)

        matrix = np.eye(4)
        matrix[:3, :3] = np.c_[x, y, z]
        nodes.append(
            pyrender.Node(
                light=pyrender.DirectionalLight(color=np.ones(3), intensity=1.0),
                matrix=matrix,
            )
        )

    return nodes


In [ ]:

class Renderer:
    def __init__(self, focal_length, faces=None):
        """
        Wrapper around the pyrender renderer to render meshes.
        Args:
            cfg (CfgNode): Model config file.
            faces (np.array): Array of shape (F, 3) containing the mesh faces.
        """
        self.focal_length = focal_length
        self.faces = faces

    def __call__(
        self,
        mesh, 
        cam_t: np.array,
        image: np.ndarray,
        full_frame: bool = False,
        imgname: Optional[str] = None,
        mesh_base_color=(1.0, 1.0, 0.9),
        scene_bg_color=(0, 0, 0),
        tri_color_lights=False,
        return_rgba=False,
        camera_center=None,
        vertex_colors=None,
    ) -> np.array:
        """
        Render meshes on input image
        Args:
            vertices (np.array): Array of shape (V, 3) containing the mesh vertices.
            cam_t (np.array): Array of shape (3,) with the camera translation.
            image (np.array): Array of (H, W, 3) containing the cropped image (unnormalized values).
            full_frame (bool): If True, then render on the full image.
            imgname (Optional[str]): Contains the original image filenamee. Used only if full_frame == True.
        """

        if full_frame:
            image = cv2.imread(imgname).astype(np.float32)
        image = image / 255.0
        h, w = image.shape[:2]

        renderer = pyrender.OffscreenRenderer(
            viewport_height=h,
            viewport_width=w,
        )

        camera_translation = cam_t.copy()
        camera_translation[0] *= -1.0

        material = pyrender.MetallicRoughnessMaterial(
            metallicFactor=0.0,
            alphaMode="OPAQUE",
            baseColorFactor=(
                mesh_base_color[2],
                mesh_base_color[1],
                mesh_base_color[0],
                1.0,
            ),  # Swap RGB to BGR for pyrender
        )

        rot = trimesh.transformations.rotation_matrix(np.radians(180), [1, 0, 0])
        mesh.apply_transform(rot)

        if vertex_colors is not None:
            mesh = pyrender.Mesh.from_trimesh(mesh)
        else:
            mesh = pyrender.Mesh.from_trimesh(mesh, material=material)

        scene = pyrender.Scene(
            bg_color=[*scene_bg_color, 0.0], ambient_light=(0.3, 0.3, 0.3)
        )
        scene.add(mesh, "mesh")

        camera_pose = np.eye(4)
        camera_pose[:3, 3] = camera_translation
        if camera_center is None:
            camera_center = [image.shape[1] / 2.0, image.shape[0] / 2.0]
        camera = pyrender.IntrinsicsCamera(
            fx=self.focal_length,
            fy=self.focal_length,
            cx=camera_center[0],
            cy=camera_center[1],
            zfar=1e12,
        )
        scene.add(camera, pose=camera_pose)

        light_nodes = create_raymond_lights()
        if tri_color_lights:
            colors = [
                np.array([1, 0.2, 0.3]),
                np.array([0.2, 1, 0.2]),
                np.array([0.2, 0.2, 1]),
            ]
            for ln, color in zip(light_nodes, colors):
                ln.light.color = color
                ln.light.intensity = 2.0

        for node in light_nodes:
            scene.add_node(node)

        color, _rend_depth = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)

        color = color.astype(np.float32) / 255.0
        renderer.delete()

        if return_rgba:
            return color

        valid_mask = (color[:, :, -1])[:, :, np.newaxis]
        output_img = color[:, :, :3] * valid_mask + (1 - valid_mask) * image

        output_img = output_img.astype(np.float32)
        return output_img

In [ ]:
renderer = Renderer(focal_length=500)


mesh_dir = "data/cow_mesh"  
cow_mesh = trimesh.load(os.path.join(mesh_dir, "cow.obj"), process=False)
mesh = pyrender.Mesh.from_trimesh(cow_mesh)

# cow_mesh.vertices[..., [1, 2]] *= -1

img = renderer(
    mesh = cow_mesh,
    cam_t = np.array([0, 0, 3]),
    image = np.ones((512, 512, 3)),
)


plt.imshow(img)
plt.show()



The rendered image is upside down because of how the image buffer is laid out.
Cause
OpenGL / pyrender use a bottom-left origin: the first row of the buffer is the bottom of the scene.
Image files and most viewers (matplotlib, OpenCV, PIL) use a top-left origin: the first row is the top of the image.
So the same 2D array is interpreted differently: what pyrender puts in row 0 (bottom) is shown as the top when you save or display it, and the image looks upside down.
Fix
Flip the image vertically after rendering:
color, _rend_depth = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)color = color[::-1, :, :].copy()  # flip so row 0 = top
or color = np.flipud(color).
Do this right after renderer.render() and before any color.astype(...) or saving. The .copy() avoids issues with negative strides if you pass the array elsewhere.
The 180° X rotation on the mesh in your Renderer is for a different reason (aligning mesh Y/Z with the camera), not for fixing the upside-down buffer; the vertical flip above is what corrects the image orientation.


In [ ]:
PATH = "/scratches/juban/cq244/BEDLAM/"
SEQ_PATH = "data/training_images/20221010_3_1000_batch01hand_6fps/png"
NPZ_PATH = os.path.join(PATH, "data/training_labels/all_npz_12_training_extra_mhr/20221010_3_1000_batch01hand_6fps.npz")

device = "cuda" if torch.cuda.is_available() else "cpu"

index = 2922

ckpt_path = "/scratch/cq244/sam-3d-body/checkpoints/sam-3d-body-dinov3/model.ckpt"
state_dict = torch.load(ckpt_path, map_location=device, weights_only=False)
faces = state_dict["head_pose.faces"].cpu().detach().numpy()

data = np.load(NPZ_PATH, allow_pickle=True)
img_path = os.path.join(PATH, SEQ_PATH, data['imgname'][index])
img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_RGB2BGR)

mhr = torch.jit.load(
    "/scratch/cq244/sam-3d-body/checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt",
    map_location=device,
)

identity_coeffs = torch.tensor(data['identity_coeffs'][index], device=device).unsqueeze(0)
lbs_model_params = torch.tensor(data['lbs_model_params'][index], device=device).unsqueeze(0)
face_expr_coeffs = torch.tensor(data['face_expr_coeffs'][index], device=device).unsqueeze(0)
mhr_output = mhr(
    identity_coeffs, lbs_model_params, face_expr_coeffs
)
verts, skeleton_state = mhr_output

verts /= 100.

mesh = trimesh.Trimesh(verts[0].cpu().detach().numpy(), faces)


focal_length = data['cam_int'][index][0, 0]
cam_int = data['cam_int'][index]
cam_t = data['cam_ext'][index][:3, 3]
cam_t += data['trans_cam'][index]
# cam_t = np.array([1.9443,  0.4980,  6.7501])

renderer = Renderer(focal_length=focal_length)

img_2 = renderer(
    mesh = mesh,
    cam_t = cam_t,
    image = img,
)
plt.imshow(img_2)
plt.show()
plt.close()
print(verts.shape)
print(cam_t.shape)
def project(points, cam_trans, cam_int):
    points = points + cam_trans
    # Normalize by Z (divide by last coordinate)
    projected_points = points / points[..., -1].unsqueeze(-1)
    # Multiply by camera intrinsics: cam_int @ projected_points.T
    projected_points = torch.einsum("bij, bkj->bki", cam_int, projected_points)
    return projected_points

verts_2d = project(
    verts, 
    torch.tensor(cam_t, device=device)[None, None, :], 
    torch.tensor(cam_int, device=device).unsqueeze(0)
)[0, :, :2]

plt.imshow(img)
plt.scatter(verts_2d[:, 0].cpu().numpy(), verts_2d[:, 1].cpu().numpy(), s=0.1, c='red')
plt.title("2D projected mesh vertices")
plt.axis("off")
plt.show()
plt.close()

print(data['imgname'][2922])
print(focal_length)
print(cam_int)
print(cam_t)
